In [86]:
spark

In [87]:
%run /opt/spark-notebooks/silver_base/parsed_data.ipynb


Master DataFrame 'parsed_df' successfully created and listening to Bronze!


In [88]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver_base")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_base.order_fact_RT (
        order_id STRING,
        order_time TIMESTAMP,
        order_status STRING,
        customer_id STRING,
        store_id STRING,
        total_items INT,
        shipping_fee DECIMAL(10,2),
        tax DECIMAL(10,2),
        currency STRING,
        platform STRING,
        app_version STRING,
        source_system STRING,
        bronze_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/base/order_fact_RT'
""")


DataFrame[]

In [89]:
# Transform into the exact shape of our Silver table using dot notation
silver_order_fact_df = parsed_df.select(
    col("data.order.order_id").alias("order_id"),
    col("data.order.order_time").cast("timestamp").alias("order_time"),
    col("data.order.status").alias("order_status"),
    col("data.order.customer.customer_id").alias("customer_id"),
    col("data.store_details.store_id").alias("store_id"),
    size(col("data.order.items")).alias("total_items"),  # Easily counting items array size
    col("data.order.charges.shipping_fee").alias("shipping_fee"),
    col("data.order.charges.tax").alias("tax"),
    col("data.order.charges.currency").alias("currency"),
    col("data.order.device_context.platform").alias("platform"),
    col("data.order.device_context.app_version").alias("app_version"),
    col("source_system"),          # Passed through from Bronze
    col("bronze_ingest_ts")        # Passed through from Bronze
)

# Start the stream!
order_fact_query = (
    silver_order_fact_df.writeStream
    .format("delta")
    .queryName("silver_order_fact_stream")
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark-data/checkpoints/silver/base/order_fact_RT/v1")
    .trigger(processingTime="30 seconds") # <-- As promised, 30s trigger for Silver Base
    .start("/opt/spark-data/delta/silver/base/order_fact_RT")
)

print("Started silver_order_fact_RT stream!")


Started silver_order_fact_RT stream!


In [ ]:
spark.sql("SELECT * FROM silver_base.order_fact_RT").show(truncate=False)


+--------+----------+------------+-----------+--------+-----------+------------+---+--------+--------+-----------+-------------+----------------+
|order_id|order_time|order_status|customer_id|store_id|total_items|shipping_fee|tax|currency|platform|app_version|source_system|bronze_ingest_ts|
+--------+----------+------------+-----------+--------+-----------+------------+---+--------+--------+-----------+-------------+----------------+
+--------+----------+------------+-----------+--------+-----------+------------+---+--------+--------+-----------+-------------+----------------+



In [91]:
spark.sql("SELECT * FROM silver_base.order_fact_RT").limit(20).toPandas()


,order_id,order_time,order_status,customer_id,store_id,total_items,shipping_fee,tax,currency,platform,app_version,source_system,bronze_ingest_ts
0,ORD-10001,2026-01-03 09:05:12,SHIPPED,C-1001,ST-001,1,500.00,2800.00,INR,ANDROID,5.2.1,order_service,2026-03-27 16:27:10.891
1,ORD-10002,2026-01-03 09:20:00,PAID,C-1002,ST-002,1,800.00,1200.00,INR,IOS,4.9.0,order_service,2026-03-27 16:27:10.891
2,ORD-10003,2026-01-03 09:40:00,SHIPPED,C-1003,ST-003,1,400.00,900.00,INR,WEB,1.0.0,order_service,2026-03-27 16:27:10.891
3,ORD-10004,2026-01-03 10:00:00,CREATED,C-1004,ST-001,1,600.00,1100.00,INR,ANDROID,5.2.2,order_service,2026-03-27 16:27:10.891
4,ORD-10005,2026-01-03 10:10:00,SHIPPED,C-1005,ST-004,1,900.00,3500.00,INR,IOS,5.0.1,order_service,2026-03-27 16:27:10.891


In [92]:
for query in spark.streams.active:
    print(query.name)

silver_order_fact_stream
silver_delivery_fact_stream
bronze_orders_raw_stream
silver_order_event_fact_stream
silver_discount_fact_stream
landing_raw_events_stream
